In [1]:
import pandas as pd

In [17]:
curve_df = pd.read_hdf('./data/data.h5', key='curve_data')
sample_info = pd.read_hdf('./data/data.h5', key='sample_info')
igi_gene_call = pd.read_hdf('./data/data.h5', key='igi_gene_call')

join_df = (curve_df
            .merge(sample_info, how='inner', on=['well_position','pcr_plate'])
            .merge(igi_gene_call, how='inner', on=['pcr_plate','sample_id','target']))
            

In [18]:
join_df.head()

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,...,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct
0,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,1,147103.828125,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
1,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,2,146864.656250,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
2,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,3,146410.109375,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
3,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,4,146188.328125,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
4,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,5,146078.421875,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined


In [19]:
join_df.columns

Index(['well_position', 'target', 'dye', 'amp_score', 'cq', 'threshold',
       'baseline_start', 'baseline_end', 'cycle_no', 'rn', 'drn', 'Fn',
       'pcr_plate', 'curve_idx', 'sample_id', 'sample_barcode', 'sample_type',
       'final_patient_result', 'current_sample_result', 'created_date',
       'record_type', 'retest_sample_id_1', 'retest_sample_id_2', 'file',
       'igi_call', 'thres_ct'],
      dtype='object')

In [35]:
clinical_ctrl_genes = (join_df
                       .loc[(join_df.sample_type == 'Clinical Sample') & (join_df.target.isin(['MS2','RnaseP'])), 
                            ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                       .copy())
clinical_ctrl_genes.loc[:,'groundtruth'] = 1

In [21]:
pos_ctrl_sample = (join_df
                   .loc[(join_df.sample_type == 'Positive Control (qPCR)'),
                        ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                   .copy())
pos_ctrl_sample.loc[:,'groundtruth'] = 1
pos_ctrl_sample.loc[pos_ctrl_sample.target.isin(['MS2','RnaseP']),'groundtruth'] = -1

In [34]:
(pos_ctrl_sample
 .groupby(['target','groundtruth'])
 .curve_idx.nunique())

target  groundtruth
E gene   1             336
MS2     -1             133
N gene   1             469
ORF1ab   1             133
RnaseP  -1             336
S gene   1             133
Name: curve_idx, dtype: int64

In [32]:
human_ctrl_sample = (join_df
                     .loc[(join_df.sample_type == 'Human Normal Negative Control (Extraction)'),
                          ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                     .copy())
human_ctrl_sample.loc[:,'groundtruth'] = -1
human_ctrl_sample.loc[human_ctrl_sample.target.isin(['MS2','RnaseP']),'groundtruth'] = 1

In [33]:
(human_ctrl_sample
 .groupby(['target','groundtruth'])
 .curve_idx.nunique())

target  groundtruth
E gene  -1             178
MS2      1              64
N gene  -1             242
ORF1ab  -1              64
RnaseP   1             178
S gene  -1              64
Name: curve_idx, dtype: int64

In [23]:
neg_ctrl_sample = (join_df
                   .loc[(join_df.sample_type == 'Negative Control (qPCR)'),
                        ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                   .copy())
neg_ctrl_sample.loc[:,'groundtruth'] = -1

In [30]:
(neg_ctrl_sample
 .groupby(['target','groundtruth'])
 .curve_idx.nunique())

target  groundtruth
E gene  -1             336
MS2     -1             133
N gene  -1             469
ORF1ab  -1             133
RnaseP  -1             336
S gene  -1             133
Name: curve_idx, dtype: int64

In [24]:
buffer_ctrl_sample = (join_df
                      .loc[(join_df.sample_type == 'Buffer Negative Control (Extraction)') & (~join_df.target.isin(['MS2','RnaseP'])),
                           ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                      .copy())
buffer_ctrl_sample.loc[:,'groundtruth'] = -1

In [28]:
(buffer_ctrl_sample
 .groupby(['target','groundtruth'])
 .curve_idx.nunique())

target  groundtruth
E gene  -1             177
N gene  -1             241
ORF1ab  -1              64
S gene  -1              64
Name: curve_idx, dtype: int64

In [37]:
groundtruth_df = pd.concat([buffer_ctrl_sample, neg_ctrl_sample, pos_ctrl_sample, 
                            human_ctrl_sample, clinical_ctrl_genes])

In [38]:
(groundtruth_df
 .groupby(['sample_type','groundtruth','target'])
 .curve_idx.nunique())

sample_type                                 groundtruth  target
Buffer Negative Control (Extraction)        -1           E gene      177
                                                         N gene      241
                                                         ORF1ab       64
                                                         S gene       64
Clinical Sample                              1           MS2        5103
                                                         RnaseP    15783
Human Normal Negative Control (Extraction)  -1           E gene      178
                                                         N gene      242
                                                         ORF1ab       64
                                                         S gene       64
                                             1           MS2          64
                                                         RnaseP      178
Negative Control (qPCR)                     -1           E g

In [41]:
groundtruth_df.to_csv('./data/groundtruth_df.csv', index = False)